# OLMo Activation Oracle Demo

This notebook shows how to use the trained activation oracle to ask natural-language questions
about the internal activations of `open_instruct_dpo_replication`.

**Flow:**
1. Load the subject model (whose activations we want to inspect)
2. Run a prompt through it and collect residual-stream activations at a chosen layer
3. Pass those activations to the oracle (same base model + oracle LoRA) via a hook at layer 1
4. Ask the oracle an arbitrary natural-language question

## 1. Imports

In [1]:
import os
os.chdir("/workspace/activation_oracles")
os.environ["TORCHDYNAMO_DISABLE"] = "1"

import torch
from peft import PeftModel

from nl_probes.utils.common import load_model, load_tokenizer, layer_percent_to_layer
from nl_probes.utils.activation_utils import collect_activations_multiple_layers, get_hf_submodule
from nl_probes.utils.dataset_utils import create_training_datapoint
from nl_probes.utils.eval import run_evaluation

/workspace/activation_oracles/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Configuration

The oracle was trained on layers at **25%, 50%, 75%** (and 88% once fully trained) of model depth.
Pick one of those for `LAYER_PERCENT`. The layer number is embedded in the oracle prompt prefix,
so it must match a layer the oracle was trained on.

In [ ]:
# SUBJECT_MODEL_NAME = "allenai/OLMo-2-0425-1B-DPO"
SUBJECT_MODEL_NAME = "allenai/OLMo-2-0425-1B-DPO"
SUBJECT_MODEL_REV  = None
TOKENIZER_NAME     = "allenai/OLMo-2-0425-1B-DPO"

ORACLE_LORA_PATH   = "downloaded_adapter/olmo2_1b_dpo_checkpoint_oracle_v1"   # downloaded from HF

LAYER_PERCENT      = 50    # which layer to inspect — must be one of [25, 50, 75]
INJECTION_LAYER    = 1     # always 1 — where the oracle hook injects activations
NUM_POSITIONS      = 10    # number of token positions to pass to the oracle

DTYPE  = torch.bfloat16
DEVICE = torch.device("cuda")

## 3. Load the model

The subject model and oracle share the same base weights. We load once and switch between
base (for activation collection) and oracle LoRA (for querying) via `disable_adapters` / `set_adapter`.

In [3]:
tokenizer  = load_tokenizer(TOKENIZER_NAME)
base_model = load_model(SUBJECT_MODEL_NAME, DTYPE, model_revision=SUBJECT_MODEL_REV)

# Load oracle LoRA — this creates a PeftModel with the oracle adapter as "default"
model = PeftModel.from_pretrained(base_model, ORACLE_LORA_PATH, is_trainable=False)

act_layer = layer_percent_to_layer(SUBJECT_MODEL_NAME, LAYER_PERCENT, SUBJECT_MODEL_REV)
print(f"Inspecting layer {act_layer} ({LAYER_PERCENT}% of model depth)")

📦 Loading tokenizer...


🧠 Loading model...


Inspecting layer 8 (50% of model depth)


## 4. Define the subject context

This is the text the subject model processes. The oracle will answer questions about
what's happening in the model's residual stream at the chosen layer while processing this text.

In [4]:
CONTEXT_MESSAGES = [
    {"role": "user", "content": "What is the capital of France?"},
]

context_text = tokenizer.apply_chat_template(
    CONTEXT_MESSAGES,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)
print("Context fed to subject model:")
print(repr(context_text))

Context fed to subject model:
'<|endoftext|><|user|>\nWhat is the capital of France?\n<|assistant|>\n'


In [5]:
# Generate the subject model's actual response
inputs_for_gen = tokenizer(
    [context_text],
    return_tensors="pt",
    add_special_tokens=False,
).to(DEVICE)

with model.disable_adapter():
    with torch.no_grad():
        output_ids = model.generate(
            **inputs_for_gen,
            do_sample=False,
            max_new_tokens=200,
            pad_token_id=tokenizer.eos_token_id,
        )

generated_ids = output_ids[0][inputs_for_gen["input_ids"].shape[1]:]
assistant_response = tokenizer.decode(generated_ids, skip_special_tokens=True)
print("Subject model response:")
print(assistant_response)

Subject model response:
The capital of France is Paris.


## 5. Collect activations from the subject model

We run the context through the base model (adapters disabled) and capture
residual-stream activations at `act_layer`.

In [6]:
inputs_BL = tokenizer(
    [context_text],
    return_tensors="pt",
    add_special_tokens=False,
).to(DEVICE)

context_ids = inputs_BL["input_ids"][0].tolist()
print(f"Context length: {len(context_ids)} tokens")

# use_lora=True traverses PeftModel correctly: model.base_model.model.model.layers[layer]
submodules = {act_layer: get_hf_submodule(model, act_layer, use_lora=True)}

# Collect activations from the base model (oracle adapter disabled)
with model.disable_adapter():
    with torch.no_grad():
        acts_by_layer = collect_activations_multiple_layers(
            model, submodules, inputs_BL, min_offset=None, max_offset=None
        )

acts_LD = acts_by_layer[act_layer][0]   # [L, D]
print(f"Activation shape: {acts_LD.shape}")

Context length: 18 tokens
Activation shape: torch.Size([18, 2048])


## 6. Ask the oracle a question

Choose which token positions to pass and what question to ask.
We take the last `NUM_POSITIONS` tokens by default (most informative for generation context).

In [7]:
QUESTION = "What is the model thinking about? What concept is most prominent?"

# Which positions to pass — last NUM_POSITIONS tokens
n_pos     = min(NUM_POSITIONS, acts_LD.shape[0])
positions = list(range(acts_LD.shape[0] - n_pos, acts_LD.shape[0]))
acts_KD   = acts_LD[positions]   # [K, D]

print(f"Passing positions {positions}")
print(f"Tokens at those positions: {[tokenizer.decode([context_ids[p]]) for p in positions]}")

Passing positions [8, 9, 10, 11, 12, 13, 14, 15, 16, 17]
Tokens at those positions: [' the', ' capital', ' of', ' France', '?\n', '<', '|', 'assistant', '|', '>\n']


In [8]:
datapoint = create_training_datapoint(
    datapoint_type="demo",
    prompt=QUESTION,
    target_response="N/A",
    layer=act_layer,
    num_positions=n_pos,
    tokenizer=tokenizer,
    acts_BD=acts_KD,
    feature_idx=-1,
    context_input_ids=None,
    context_positions=None,
)

injection_submodule = get_hf_submodule(model, INJECTION_LAYER, use_lora=True)

responses = run_evaluation(
    eval_data=[datapoint],
    model=model,
    tokenizer=tokenizer,
    submodule=injection_submodule,
    device=DEVICE,
    dtype=DTYPE,
    global_step=-1,
    lora_path=None,   # oracle adapter already loaded as "default"
    eval_batch_size=1,
    steering_coefficient=1.0,
    generation_kwargs={"do_sample": False, "max_new_tokens": 100},
)

print("Oracle response:")
print(responses[0].api_response)


Evaluating model:   0%|          | 0/1 [00:00<?, ?it/s]


Evaluating model: 100%|██████████| 1/1 [00:00<00:00,  1.80it/s]


Evaluating model: 100%|██████████| 1/1 [00:00<00:00,  1.80it/s]

Oracle response:
The assistant is most likely considering the concept of time and its relationship to the passage of years.


## 7. Sweep over individual token positions

Query the oracle once per token — each time passing a single token's activation.
This lets you see which positions carry the most interesting signal.

In [9]:
QUESTION_PER_TOKEN = "What concept or word is most salient at this position?"

per_token_datapoints = []
for i in range(len(context_ids)):
    dp = create_training_datapoint(
        datapoint_type="demo",
        prompt=QUESTION_PER_TOKEN,
        target_response="N/A",
        layer=act_layer,
        num_positions=1,
        tokenizer=tokenizer,
        acts_BD=acts_LD[i:i+1],
        feature_idx=i,
        context_input_ids=None,
        context_positions=None,
    )
    per_token_datapoints.append(dp)

per_token_responses = run_evaluation(
    eval_data=per_token_datapoints,
    model=model,
    tokenizer=tokenizer,
    submodule=injection_submodule,
    device=DEVICE,
    dtype=DTYPE,
    global_step=-1,
    lora_path=None,   # oracle adapter already loaded as "default"
    eval_batch_size=16,
    steering_coefficient=1.0,
    generation_kwargs={"do_sample": False, "max_new_tokens": 30},
)

print(f"{'Pos':>4}  {'Token':<20}  Response")
print("-" * 70)
for i, resp in enumerate(per_token_responses):
    tok = tokenizer.decode([context_ids[i]]).replace("\n", "\\n")
    print(f"{i:>4}  {tok:<20}  {resp.api_response}")


Evaluating model:   0%|          | 0/2 [00:00<?, ?it/s]


Evaluating model:  50%|█████     | 1/2 [00:02<00:02,  2.04s/it]


Evaluating model: 100%|██████████| 2/2 [00:02<00:00,  1.05s/it]


Evaluating model: 100%|██████████| 2/2 [00:02<00:00,  1.20s/it]

 Pos  Token                 Response
----------------------------------------------------------------------
   0  <|endoftext|>         The concept of 'reinforcement learning' is most salient at this position.
   1  <                     The concept of 'reinforcement learning' is most salient at this position.
   2  |                     The concept of 'reinforcement' is most salient at this position.
   3  user                  The concept of 'reinforcement learning' is most salient at this position.
   4  |                     The concept of 'reinforcement' is most salient at this position.
   5  >\n                   The concept of 'reinforcement' is most salient at this position.
   6  What                  The concept of 'reification' is most salient at this position.
   7   is                   The concept of 'reification' is most salient at this position.
   8   the                  The concept of 'reification' is most salient at this position.
   9   capital              The co

## 8. Sweep over prompt + response tokens

Repeat the per-token sweep but now over the **full sequence** — user prompt followed by
the model's actual response. Tokens are labelled `[P]` (prompt) or `[R]` (response)
so you can see how the activations evolve as the model generates its answer.

In [10]:
# Build full conversation text including the assistant's response
full_text = tokenizer.apply_chat_template(
    CONTEXT_MESSAGES + [{"role": "assistant", "content": assistant_response}],
    tokenize=False,
    add_generation_prompt=False,
    enable_thinking=False,
)

full_inputs = tokenizer(
    [full_text],
    return_tensors="pt",
    add_special_tokens=False,
).to(DEVICE)

full_ids = full_inputs["input_ids"][0].tolist()
prompt_len = len(context_ids)   # split point between prompt and response tokens

print(f"Prompt tokens:   {prompt_len}")
print(f"Response tokens: {len(full_ids) - prompt_len}")
print(f"Total tokens:    {len(full_ids)}")

# Collect activations over the full sequence
full_submodules = {act_layer: get_hf_submodule(model, act_layer, use_lora=True)}
with model.disable_adapter():
    with torch.no_grad():
        full_acts_by_layer = collect_activations_multiple_layers(
            model, full_submodules, full_inputs, min_offset=None, max_offset=None
        )

full_acts_LD = full_acts_by_layer[act_layer][0]   # [L, D]
print(f"Activation shape: {full_acts_LD.shape}")

# Build one oracle datapoint per token
full_per_token_datapoints = []
for i in range(len(full_ids)):
    dp = create_training_datapoint(
        datapoint_type="demo",
        prompt=QUESTION_PER_TOKEN,
        target_response="N/A",
        layer=act_layer,
        num_positions=1,
        tokenizer=tokenizer,
        acts_BD=full_acts_LD[i:i+1],
        feature_idx=i,
        context_input_ids=None,
        context_positions=None,
    )
    full_per_token_datapoints.append(dp)

full_per_token_responses = run_evaluation(
    eval_data=full_per_token_datapoints,
    model=model,
    tokenizer=tokenizer,
    submodule=injection_submodule,
    device=DEVICE,
    dtype=DTYPE,
    global_step=-1,
    lora_path=None,
    eval_batch_size=16,
    steering_coefficient=1.0,
    generation_kwargs={"do_sample": False, "max_new_tokens": 30},
)

print(f"\n{'Pos':>4}  {'':3}  {'Token':<22}  Response")
print("-" * 80)
for i, resp in enumerate(full_per_token_responses):
    label = "[P]" if i < prompt_len else "[R]"
    tok = tokenizer.decode([full_ids[i]]).replace("\n", "\\n")
    print(f"{i:>4}  {label}  {tok:<22}  {resp.api_response}")

Prompt tokens:   18
Response tokens: 8
Total tokens:    26
Activation shape: torch.Size([26, 2048])



Evaluating model:   0%|          | 0/2 [00:00<?, ?it/s]


Evaluating model:  50%|█████     | 1/2 [00:00<00:00,  2.52it/s]


Evaluating model: 100%|██████████| 2/2 [00:00<00:00,  2.79it/s]


Evaluating model: 100%|██████████| 2/2 [00:00<00:00,  2.74it/s]


 Pos       Token                   Response
--------------------------------------------------------------------------------
   0  [P]  <|endoftext|>           The concept of 'reinforcement learning' is most salient at this position.
   1  [P]  <                       The concept of 'reinforcement learning' is most salient at this position.
   2  [P]  |                       The concept of 'reinforcement' is most salient at this position.
   3  [P]  user                    The concept of 'reinforcement learning' is most salient at this position.
   4  [P]  |                       The concept of 'reinforcement' is most salient at this position.
   5  [P]  >\n                     The concept of 'reinforcement' is most salient at this position.
   6  [P]  What                    The concept of 'reification' is most salient at this position.
   7  [P]   is                     The concept of 'reification' is most salient at this position.
   8  [P]   the                    The concept of '

## 9. Ask the oracle: what is the goal of the model?

Using the same activations collected from the prompt, ask the oracle a higher-level question
about the model's overall intent.

In [11]:
goal_datapoint = create_training_datapoint(
    datapoint_type="demo",
    prompt="What is the goal of the model?",
    target_response="N/A",
    layer=act_layer,
    num_positions=n_pos,
    tokenizer=tokenizer,
    acts_BD=acts_KD,
    feature_idx=-1,
    context_input_ids=None,
    context_positions=None,
)

goal_responses = run_evaluation(
    eval_data=[goal_datapoint],
    model=model,
    tokenizer=tokenizer,
    submodule=injection_submodule,
    device=DEVICE,
    dtype=DTYPE,
    global_step=-1,
    lora_path=None,
    eval_batch_size=1,
    steering_coefficient=1.0,
    generation_kwargs={"do_sample": False, "max_new_tokens": 100},
)

print("Oracle response:")
print(goal_responses[0].api_response)


Evaluating model:   0%|          | 0/1 [00:00<?, ?it/s]


Evaluating model: 100%|██████████| 1/1 [00:00<00:00,  2.82it/s]


Evaluating model: 100%|██████████| 1/1 [00:00<00:00,  2.81it/s]

Oracle response:
The goal of the model is to provide accurate and concise information about the capital of France.
